# Colab Launcher — E2VID Reconstruction + YOLO Training

This notebook is a thin wrapper that runs the two Python scripts on Colab GPU.

**Before running:**
1. Upload your `ami/` folder (or at least `data/` and `scripts/`) to Google Drive
2. Edit the **Configuration** cell below
3. Run all cells top to bottom

**Output:** `data/yolo_e2vid.pt` — copy to `services/e2vid/weights/yolo_e2vid.pt` and rebuild Docker.

## 1 · Mount Google Drive

In [ ]:
import os, shutil, subprocess
from google.colab import drive

# Unmount if already mounted (FUSE), then clear the directory
subprocess.run(['fusermount', '-u', '/content/drive'], capture_output=True)
if os.path.isdir('/content/drive') and os.listdir('/content/drive'):
    shutil.rmtree('/content/drive')

drive.mount('/content/drive')

## 2 · Configuration — edit this cell

In [ ]:
from pathlib import Path
import datetime

# ── Root of your ami/ folder on Google Drive ──────────────────────────────────
AMI_ROOT = Path('/content/drive/MyDrive/ami')

# ── Resume a previous run that was interrupted? ───────────────────────────────
# Set to True to skip reconstruction cleanup and resume training from last.pt.
# The dataset is always rebuilt from scratch (it lives on local SSD, not Drive).
RESUME = False

# ── Sequences ─────────────────────────────────────────────────────────────────
SEQUENCES     = ['sequence_0', 'sequence_1', 'sequence_2', 'sequence_3', 'sequence_8']
VAL_SEQUENCES = ['sequence_8']

# ── Reconstruction parameters ─────────────────────────────────────────────────
START_S          = 5.0
EVENTS_PER_PIXEL = 0.01
SMOKE_EVENTS     = None   # e.g. 100_000 for a quick test

# ── Training parameters ───────────────────────────────────────────────────────
EPOCHS = 100
BATCH  = 16

# ── Derived paths ─────────────────────────────────────────────────────────────
RAW_ROOT    = AMI_ROOT / 'data' / 'raw'
RECON_ROOT  = AMI_ROOT / 'data' / 'processed'
WEIGHTS_OUT = AMI_ROOT / 'data' / 'yolo_e2vid.pt'       # Drive — persistent
RUNS_DIR    = AMI_ROOT / 'data' / 'yolo_runs'            # Drive — checkpoints persist across sessions
SCRIPTS_DIR = AMI_ROOT / 'scripts'
LOG_FILE    = AMI_ROOT / 'logs' / f'run_{datetime.datetime.now().strftime("%Y%m%d_%H%M%S")}.log'

# Dataset on local SSD — fast random I/O during training. Rebuilt each session.
DATASET_DIR = Path('/content/yolo_e2vid')

WORK_DIR = Path('/content/work')
WORK_DIR.mkdir(exist_ok=True)
LOG_FILE.parent.mkdir(parents=True, exist_ok=True)

train_seqs = [s for s in SEQUENCES if s not in VAL_SEQUENCES]
print('AMI root      :', AMI_ROOT)
print('Train seqs    :', train_seqs)
print('Val seqs      :', VAL_SEQUENCES)
print('Resume        :', RESUME)
print('Dataset dir   :', DATASET_DIR, '← local SSD')
print('Runs dir      :', RUNS_DIR, '← Drive (checkpoints)')
print('Smoke events  :', SMOKE_EVENTS or 'full run')
print('Epochs        :', EPOCHS)
print('Log file      :', LOG_FILE)

## 3 · Install dependencies

In [ ]:
!pip install -q h5py ultralytics==8.4.54 imageio scikit-image pandas matplotlib
import torch
print(f'PyTorch {torch.__version__} — CUDA: {torch.cuda.is_available()}')

In [ ]:
import torch
if not torch.cuda.is_available():
    raise SystemExit(
        'No GPU detected. Go to Runtime → Change runtime type → '
        'Hardware accelerator → T4 GPU, then reconnect and re-run.'
    )
print(f'GPU: {torch.cuda.get_device_name(0)}  '
      f'({torch.cuda.get_device_properties(0).total_memory // 1024**2} MB)')

## 5 · Reconstruct e2vid frames

In [ ]:
import os, subprocess, sys, datetime, shutil

# Copy scripts from Drive to local SSD to avoid Drive cache/sync issues
LOCAL_SCRIPTS = Path('/content/scripts')
LOCAL_SCRIPTS.mkdir(exist_ok=True)
for script in ['reconstruct.py', 'train_yolo.py']:
    shutil.copy(SCRIPTS_DIR / script, LOCAL_SCRIPTS / script)
print(f'Scripts copied to {LOCAL_SCRIPTS}')

def run_streaming(cmd):
    """Run a command, stream output to notebook and append to log file on Drive."""
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    with open(LOG_FILE, 'a') as lf:
        for line in process.stdout:
            print(line, end='', flush=True)
            lf.write(line)
            lf.flush()
    process.wait()
    return process.returncode

def log(msg):
    ts = datetime.datetime.now().strftime('%H:%M:%S')
    line = f'[{ts}] {msg}'
    print(line)
    with open(LOG_FILE, 'a') as f:
        f.write(line + '\n')

log('Helpers ready.')

## 4 · Cleanup — remove stale output before each run

In [ ]:
import shutil

if RESUME:
    print('RESUME=True — skipping cleanup, keeping existing outputs.')
else:
    # Remove stale reconstruction output so the skip-check doesn't trigger
    # and no leftover frames from a previous run pollute the new one.
    for seq in SEQUENCES:
        recon_dir = RECON_ROOT / seq / 'reconstruction_e2vid'
        if recon_dir.exists():
            shutil.rmtree(recon_dir)
            print(f'Cleaned: {recon_dir}')
        else:
            print(f'Nothing to clean: {recon_dir}')

    shutil.rmtree(DATASET_DIR, ignore_errors=True)
    print(f'Cleaned: {DATASET_DIR}')

    # Remove cached rpg_e2vid clone so patches always re-apply from scratch
    shutil.rmtree(WORK_DIR / 'rpg_e2vid', ignore_errors=True)
    print(f'Cleaned: {WORK_DIR / "rpg_e2vid"}')

In [ ]:
log('=== Reconstruction started ===')

for seq in SEQUENCES:
    zip_path = RECON_ROOT / seq / 'events.zip'
    out_dir  = RECON_ROOT / seq / 'reconstruction_e2vid'

    if out_dir.exists() and any(out_dir.glob('frame_*.png')):
        log(f'{seq}: frames already exist — skipping reconstruction')
        continue

    log(f'=== Reconstructing {seq} ===')
    cmd = [
        sys.executable, str(LOCAL_SCRIPTS / 'reconstruct.py'),
        '--zip_path',         str(zip_path),
        '--out_dir',          str(out_dir),
        '--work_dir',         str(WORK_DIR),
        '--events_per_pixel', str(EVENTS_PER_PIXEL),
    ]
    if SMOKE_EVENTS:
        cmd += ['--max_events', str(SMOKE_EVENTS)]

    rc = run_streaming(cmd)
    if rc != 0:
        log(f'ERROR: reconstruct.py failed for {seq} (exit code {rc})')
        raise RuntimeError(f'reconstruct.py failed for {seq} (exit code {rc})')

log('=== Reconstruction done ===')

## 6 · Train YOLO

In [ ]:
log('=== Training started ===')

# When resuming, the checkpoint stores the old dataset.yaml path (on Drive).
# Overwrite that yaml to redirect ultralytics to the local SSD dataset instead.
# This must happen AFTER the dataset is built locally (step 2 in train_yolo.py),
# so we write it here before the subprocess runs — train_yolo.py will rebuild it
# locally, but the Drive yaml is what the resume checkpoint will read.
drive_yaml = AMI_ROOT / 'data' / 'yolo_e2vid' / 'dataset.yaml'
drive_yaml.parent.mkdir(parents=True, exist_ok=True)
drive_yaml.write_text(f"""path: {DATASET_DIR}
train: images/train
val:   images/val

nc: 1
names:
  0: drone
""")
log(f'Drive dataset.yaml updated → {DATASET_DIR} (local SSD)')

cmd = [
    sys.executable, str(LOCAL_SCRIPTS / 'train_yolo.py'),
    '--sequences',  *SEQUENCES,
    '--raw_root',   str(RAW_ROOT),
    '--recon_root', str(RECON_ROOT),
    '--out_dir',    str(DATASET_DIR),
    '--runs_dir',   str(RUNS_DIR),
    '--weights',    str(WEIGHTS_OUT),
    '--epochs',     str(EPOCHS),
    '--batch',      str(BATCH),
]

if VAL_SEQUENCES:
    cmd += ['--val_sequences', *VAL_SEQUENCES]

if RESUME:
    cmd += ['--resume']

rc = run_streaming(cmd)
if rc != 0:
    log(f'ERROR: train_yolo.py failed (exit code {rc})')
    raise RuntimeError('train_yolo.py failed')

log(f'=== Training done — weights at {WEIGHTS_OUT} ===')
print('Download and copy to services/e2vid/weights/yolo_e2vid.pt')

## 7 · Download results to your laptop

Results are already on Google Drive. On your laptop run:

```bash
bash ~/ami/scripts/sync_from_drive.sh
```